# pandas
## 데이터 분석 입문자를 위한 표 데이터 처리 교안

> NumPy 교안을 먼저 보시면 좋습니다. pandas는 **NumPy 배열 위에 이름표를 붙인 것**이라, 배열 개념이 그대로 이어집니다.

### 학습 방법

- 셀을 **직접 실행**하고 값을 바꿔 보세요.
- 결과가 이상하면 **`.shape` 과 `.dtypes` 부터** 확인하세요.


---

# Part 0. pandas란?

## 0.1 한 줄 요약

> **엑셀 표를 파이썬 코드로 다루는 라이브러리.**

행과 열에 **이름(라벨)** 이 붙은 표를 다룹니다. 그래서 `df[3][7]` 이 아니라 `df.loc['서울', '인구']` 처럼 쓸 수 있습니다.

## 0.2 NumPy와 무엇이 다른가

| | NumPy `ndarray` | pandas `DataFrame` |
|---|---|---|
| 라벨 | 없음 (숫자 인덱스만) | **행·열 이름이 있음** |
| 타입 | 전체가 한 타입 | **열마다 다른 타입 가능** |
| 결측치 | 지원 빈약 | **`NaN` / `NA` 전용 기능** |
| 그룹 집계 | 직접 구현 | **`groupby` 한 줄** |
| 파일 입출력 | 기본적 | **CSV·엑셀·SQL·JSON** |
| 속도 | 더 빠름 | 조금 느림 (대신 편함) |

```
     ┌──────────── DataFrame ────────────┐
     │        이름   나이   도시          │  ← 열 이름 (columns)
     │  0     철수    25    서울          │
     │  1     영희    31    부산          │
     │  2     민수    28    대구          │
     │  ↑                                 │
     │  행 이름 (index)                   │
     └────────────────────────────────────┘
            각 열 = Series (1차원)
```

## 0.3 언제 무엇을 쓰나

| 상황 | 도구 |
|---|---|
| 순수 수치 계산, 행렬 연산 | NumPy |
| 열마다 타입이 다른 표 | **pandas** |
| 결측치·문자열·날짜가 섞인 실제 데이터 | **pandas** |
| 그룹별 집계, 조인 | **pandas** |
| 메모리를 넘는 대용량 | Polars, DuckDB, Dask |

In [1]:
import numpy as np
import pandas as pd

print('pandas', pd.__version__)
print('numpy ', np.__version__)

# 보기 좋게 출력하는 설정 (선택)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

pandas 2.2.2
numpy  2.0.2


---

# Part 1. Series와 DataFrame

## 1.1 Series — 이름표가 붙은 1차원 배열

```
  index    values
   서울  →   941
   부산  →   334
   대구  →   238
```

In [2]:
s = pd.Series([941, 334, 238, 297],
              index=['서울', '부산', '대구', '인천'],
              name='인구')
print(s)
print()
print('values :', s.values, type(s.values))     # 속은 NumPy 배열
print('index  :', list(s.index))
print('dtype  :', s.dtype)
print('name   :', s.name)
print()
print('서울 값 :', s['서울'])
print('300 이상:\n', s[s >= 300])

서울    941
부산    334
대구    238
인천    297
Name: 인구, dtype: int64

values : [941 334 238 297] <class 'numpy.ndarray'>
index  : ['서울', '부산', '대구', '인천']
dtype  : int64
name   : 인구

서울 값 : 941
300 이상:
 서울    941
부산    334
Name: 인구, dtype: int64


## 1.2 DataFrame — Series를 열로 묶은 2차원 표

만드는 방법이 여러 가지입니다. **딕셔너리 방식**이 가장 흔합니다.

In [3]:
# 방법 1: 딕셔너리 (열 이름 → 값 리스트)  ← 가장 흔함
df = pd.DataFrame({
    '이름': ['철수', '영희', '민수', '지은', '현우'],
    '나이': [25, 31, 28, 35, 22],
    '도시': ['서울', '부산', '서울', '대구', '부산'],
    '점수': [88.5, 92.0, 79.5, 95.0, 67.0],
})
print(df)

# 방법 2: 리스트의 리스트 + columns
df2 = pd.DataFrame([['철수', 25], ['영희', 31]], columns=['이름', '나이'])

# 방법 3: NumPy 배열
df3 = pd.DataFrame(np.arange(6).reshape(2, 3), columns=list('ABC'))

# 방법 4: 딕셔너리의 리스트 (레코드 방식)
df4 = pd.DataFrame([{'이름': '철수', '나이': 25}, {'이름': '영희', '나이': 31}])

print('\n방법 3\n', df3)

   이름  나이  도시    점수
0  철수  25  서울 88.50
1  영희  31  부산 92.00
2  민수  28  서울 79.50
3  지은  35  대구 95.00
4  현우  22  부산 67.00

방법 3
    A  B  C
0  0  1  2
1  3  4  5


In [4]:
# DataFrame의 구성 요소
print('shape   :', df.shape)          # (행, 열)
print('columns :', list(df.columns))
print('index   :', list(df.index))
print('dtypes  :\n', df.dtypes)
print()
print('열 하나를 꺼내면 Series :', type(df['이름']))
print('열 여러 개는 DataFrame  :', type(df[['이름', '나이']]))
print()
print('NumPy 배열로 :', df[['나이', '점수']].to_numpy().shape)

shape   : (5, 4)
columns : ['이름', '나이', '도시', '점수']
index   : [0, 1, 2, 3, 4]
dtypes  :
 이름     object
나이      int64
도시     object
점수    float64
dtype: object

열 하나를 꺼내면 Series : <class 'pandas.core.series.Series'>
열 여러 개는 DataFrame  : <class 'pandas.core.frame.DataFrame'>

NumPy 배열로 : (5, 2)


## 1.3 인덱스 다루기

인덱스는 **행의 이름표**입니다. 기본은 0,1,2...이지만 의미 있는 값으로 바꿀 수 있습니다.

In [5]:
d = df.set_index('이름')          # '이름' 열을 인덱스로
print(d)
print('\n영희 행:\n', d.loc['영희'])

print('\nreset_index — 인덱스를 다시 열로:\n', d.reset_index().head(2))
print('\nrename_axis:', d.rename_axis('학생').index.name)

    나이  도시    점수
이름              
철수  25  서울 88.50
영희  31  부산 92.00
민수  28  서울 79.50
지은  35  대구 95.00
현우  22  부산 67.00

영희 행:
 나이      31
도시      부산
점수   92.00
Name: 영희, dtype: object

reset_index — 인덱스를 다시 열로:
    이름  나이  도시    점수
0  철수  25  서울 88.50
1  영희  31  부산 92.00

rename_axis: 학생


## 연습문제 1

**1-1.** 과일 이름을 인덱스로, 가격을 값으로 하는 Series를 만드세요.

**1-2.** 딕셔너리로 `상품 / 가격 / 재고` 3열짜리 DataFrame을 5행 만드세요.

**1-3.** 위 DataFrame의 `shape`, `columns`, `dtypes` 를 출력하세요.

**1-4.** `상품` 을 인덱스로 설정한 뒤 다시 열로 되돌리세요.

---

# Part 2. 데이터 불러오기 · 저장하기

## 2.1 주요 함수

| 읽기 | 쓰기 | 형식 |
|---|---|---|
| `pd.read_csv()` | `df.to_csv()` | CSV |
| `pd.read_excel()` | `df.to_excel()` | 엑셀 (`openpyxl` 필요) |
| `pd.read_json()` | `df.to_json()` | JSON |
| `pd.read_parquet()` | `df.to_parquet()` | Parquet (**대용량에 최적**) |
| `pd.read_sql()` | `df.to_sql()` | 데이터베이스 |
| `pd.read_clipboard()` | `df.to_clipboard()` | 클립보드 |

## 2.2 `read_csv` 자주 쓰는 옵션

| 옵션 | 뜻 |
|---|---|
| `encoding='utf-8'` / `'cp949'` | **한글 깨질 때 `cp949` 시도** |
| `sep=','` / `'\t'` | 구분자 |
| `header=None` | 첫 줄이 데이터일 때 |
| `names=[...]` | 열 이름 직접 지정 |
| `index_col=0` | 특정 열을 인덱스로 |
| `usecols=['a','b']` | 필요한 열만 (**메모리 절약**) |
| `dtype={'id': str}` | 타입 지정 (앞자리 0 보존) |
| `parse_dates=['date']` | 날짜로 변환 |
| `na_values=['-', 'N/A']` | 결측으로 취급할 값 |
| `nrows=1000` | 앞부분만 (**구조 확인용**) |
| `thousands=','` | 천단위 콤마 제거 |
| `skiprows=3` | 위 몇 줄 건너뛰기 |

In [6]:
# 실습용 데이터를 직접 만듭니다 (인터넷 불필요)
rng = np.random.default_rng(42)
n = 800

sales = pd.DataFrame({
    '주문일': pd.to_datetime('2024-01-01') + pd.to_timedelta(rng.integers(0, 365, n), unit='D'),
    '지역': rng.choice(['수도권', '영남', '호남', '충청', '강원'], n, p=[.40, .24, .16, .15, .05]),
    '카테고리': rng.choice(['가전', '의류', '식품', '도서'], n, p=[.20, .30, .35, .15]),
    '채널': rng.choice(['온라인', '오프라인'], n, p=[.65, .35]),
    '고객ID': rng.integers(1000, 1200, n),
    '수량': rng.integers(1, 8, n),
})
base_price = {'가전': 250000, '의류': 45000, '식품': 15000, '도서': 18000}
sales['단가'] = (sales['카테고리'].map(base_price) * rng.normal(1, 0.15, n)).round(0)
sales['금액'] = sales['단가'] * sales['수량']

# 일부러 결측치와 중복을 섞습니다 (정리 연습용)
sales.loc[rng.choice(n, 30, replace=False), '지역'] = np.nan
sales.loc[rng.choice(n, 15, replace=False), '단가'] = np.nan
sales = pd.concat([sales, sales.sample(10, random_state=1)], ignore_index=True)

sales.to_csv('sales.csv', index=False, encoding='utf-8-sig')
print('sales.csv 생성:', sales.shape)
sales.head()

sales.csv 생성: (810, 8)


,주문일,지역,카테고리,채널,고객ID,수량,단가,금액
0,2024-02-02,수도권,식품,오프라인,1067,4,"16,005.00","64,020.00"
1,2024-10-09,충청,식품,온라인,1199,6,"14,865.00","89,190.00"
2,2024-08-26,호남,식품,오프라인,1001,3,"15,028.00","45,084.00"
3,2024-06-09,수도권,식품,오프라인,1098,7,"16,478.00","115,346.00"
4,2024-06-07,강원,가전,온라인,1121,6,"222,724.00","1,336,344.00"


In [7]:
# 다시 읽어오기
df = pd.read_csv('sales.csv', parse_dates=['주문일'], encoding='utf-8-sig')
print(df.shape)
print(df.dtypes)

# 구조만 빠르게 확인할 때
peek = pd.read_csv('sales.csv', nrows=5)
print('\n앞 5줄만:', peek.shape)

# 필요한 열만 (대용량일 때 메모리 절약)
light = pd.read_csv('sales.csv', usecols=['주문일', '지역', '금액'])
print('일부 열만:', light.shape)

(810, 8)
주문일     datetime64[ns]
지역              object
카테고리            object
채널              object
고객ID             int64
수량               int64
단가             float64
금액             float64
dtype: object

앞 5줄만: (5, 8)
일부 열만: (810, 3)


> **한글 CSV 인코딩**
> - 엑셀에서 만든 CSV → `encoding='cp949'` 또는 `'euc-kr'`
> - 저장할 때 엑셀에서 한글이 깨지면 → `to_csv(..., encoding='utf-8-sig')`
> - `UnicodeDecodeError` 가 뜨면 인코딩부터 바꿔 보세요.

---

# Part 3. 데이터 살펴보기

**새 데이터를 받으면 항상 이 순서로 훑습니다.**

| 함수 | 무엇을 보나 |
|---|---|
| `df.head()` / `.tail()` | 실제 값 생김새 |
| `df.shape` | 행·열 개수 |
| `df.info()` | **타입 + 결측 개수 + 메모리** (가장 중요) |
| `df.describe()` | 수치형 요약 통계 |
| `df.describe(include='object')` | 문자형 요약 |
| `df.isna().sum()` | 열별 결측 개수 |
| `df['x'].value_counts()` | 범주별 빈도 |
| `df.nunique()` | 열별 고유값 개수 |
| `df.duplicated().sum()` | 중복 행 개수 |

In [8]:
print('=== shape ===');  print(df.shape)
print('\n=== head ===');  print(df.head(3))
print('\n=== info ===');  df.info()

=== shape ===
(810, 8)

=== head ===
         주문일   지역 카테고리    채널  고객ID  수량        단가        금액
0 2024-02-02  수도권   식품  오프라인  1067   4 16,005.00 64,020.00
1 2024-10-09   충청   식품   온라인  1199   6 14,865.00 89,190.00
2 2024-08-26   호남   식품  오프라인  1001   3 15,028.00 45,084.00

=== info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 810 entries, 0 to 809
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   주문일     810 non-null    datetime64[ns]
 1   지역      780 non-null    object        
 2   카테고리    810 non-null    object        
 3   채널      810 non-null    object        
 4   고객ID    810 non-null    int64         
 5   수량      810 non-null    int64         
 6   단가      795 non-null    float64       
 7   금액      810 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(3)
memory usage: 50.8+ KB


In [9]:
print('=== 수치형 요약 ===')
print(df.describe().round(1))

print('\n=== 문자형 요약 ===')
print(df.describe(include='object'))

print('\n=== 결측 개수 ===')
print(df.isna().sum())

print('\n=== 결측 비율 ===')
print((df.isna().mean() * 100).round(1))

=== 수치형 요약 ===
                                 주문일     고객ID     수량         단가           금액
count                            810   810.00 810.00     795.00       810.00
mean   2024-07-03 12:14:13.333333248 1,095.50   4.00  70,453.60   273,634.70
min              2024-01-02 00:00:00 1,000.00   1.00   9,558.00    10,638.00
25%              2024-04-05 00:00:00 1,044.00   2.00  15,997.00    54,273.00
50%              2024-07-03 00:00:00 1,093.00   4.00  33,019.00   103,519.50
75%              2024-10-04 18:00:00 1,143.00   6.00  49,728.50   253,782.20
max              2024-12-30 00:00:00 1,199.00   7.00 379,527.00 2,656,689.00
std                              NaN    57.50   2.00  90,912.70   418,147.80

=== 문자형 요약 ===
         지역 카테고리   채널
count   780  810  810
unique    5    4    2
top     수도권   식품  온라인
freq    313  265  532

=== 결측 개수 ===
주문일      0
지역      30
카테고리     0
채널       0
고객ID     0
수량       0
단가      15
금액       0
dtype: int64

=== 결측 비율 ===
주문일    0.00
지역     3.70
카테고리   0.00

In [10]:
print('=== 범주별 빈도 ===')
print(df['카테고리'].value_counts())
print('\n비율:')
print(df['카테고리'].value_counts(normalize=True).round(3))

print('\n=== 두 열 교차 빈도 ===')
print(pd.crosstab(df['지역'], df['채널']))

print('\n=== 고유값 개수 ===')
print(df.nunique())

print('\n=== 중복 행 ===', df.duplicated().sum(), '개')

=== 범주별 빈도 ===
카테고리
식품    265
의류    259
가전    154
도서    132
Name: count, dtype: int64

비율:
카테고리
식품   0.33
의류   0.32
가전   0.19
도서   0.16
Name: proportion, dtype: float64

=== 두 열 교차 빈도 ===
채널   오프라인  온라인
지역            
강원     21   24
수도권   112  201
영남     54  124
충청     42   75
호남     38   89

=== 고유값 개수 ===
주문일     320
지역        5
카테고리      4
채널        2
고객ID    196
수량        7
단가      779
금액      799
dtype: int64

=== 중복 행 === 10 개


## 연습문제 3

**3-1.** `df.info()` 를 실행해 결측이 있는 열이 무엇인지 찾으세요.

**3-2.** `지역` 열의 값별 빈도와 비율을 구하세요.

**3-3.** `카테고리` × `채널` 교차표를 만드세요.

**3-4.** 열별 결측 비율을 퍼센트로 출력하세요.

---

# Part 4. 선택하기 — `loc`, `iloc`, 조건

**pandas에서 가장 많이 쓰고 가장 많이 헷갈리는 부분입니다.**

## 4.1 세 가지 방식

| 방식 | 기준 | 예 |
|---|---|---|
| `df['열']` | **열 이름** | `df['금액']` |
| `df.loc[행라벨, 열라벨]` | **라벨(이름)** | `df.loc[0:5, '지역':'채널']` |
| `df.iloc[행번호, 열번호]` | **위치(정수)** | `df.iloc[0:5, 1:4]` |

> **결정적 차이:** `loc` 의 슬라이스는 **끝을 포함**하고, `iloc` 은 **끝을 제외**합니다.
> `df.loc[0:5]` → 6개 행 / `df.iloc[0:5]` → 5개 행

In [11]:
print('df["금액"] 앞 3개:\n', df['금액'].head(3))
print('\n여러 열:\n', df[['지역', '금액']].head(3))

print('\nloc[0:3] →', len(df.loc[0:3]), '행 (끝 포함)')
print('iloc[0:3] →', len(df.iloc[0:3]), '행 (끝 제외)')

print('\nloc 행+열:\n', df.loc[0:2, ['지역', '카테고리', '금액']])
print('\niloc 행+열:\n', df.iloc[0:2, 1:4])
print('\n스칼라 하나:', df.loc[0, '금액'], '=', df.iat[0, df.columns.get_loc('금액')])

df["금액"] 앞 3개:
 0   64,020.00
1   89,190.00
2   45,084.00
Name: 금액, dtype: float64

여러 열:
     지역        금액
0  수도권 64,020.00
1   충청 89,190.00
2   호남 45,084.00

loc[0:3] → 4 행 (끝 포함)
iloc[0:3] → 3 행 (끝 제외)

loc 행+열:
     지역 카테고리        금액
0  수도권   식품 64,020.00
1   충청   식품 89,190.00
2   호남   식품 45,084.00

iloc 행+열:
     지역 카테고리    채널
0  수도권   식품  오프라인
1   충청   식품   온라인

스칼라 하나: 64020.0 = 64020.0


## 4.2 조건으로 고르기 (불리언 인덱싱)

**실무에서 가장 많이 씁니다.** NumPy와 규칙이 같습니다 — `&`, `|`, `~` 와 **괄호**.

In [12]:
high = df[df['금액'] >= 500_000]
print('50만원 이상 주문:', len(high), '건')

# 조건 여러 개 — 각 조건을 괄호로!
cond = df[(df['카테고리'] == '가전') & (df['채널'] == '온라인')]
print('가전 + 온라인   :', len(cond), '건')

cond2 = df[(df['지역'] == '수도권') | (df['지역'] == '영남')]
print('수도권 또는 영남:', len(cond2), '건')

# isin — 여러 값 중 하나
print('isin 사용       :', len(df[df['지역'].isin(['수도권', '영남'])]), '건')

# between — 범위
print('수량 3~5개      :', len(df[df['수량'].between(3, 5)]), '건')

# 부정
print('가전이 아닌 것  :', len(df[~(df['카테고리'] == '가전')]), '건')

# 결측 여부
print('지역이 결측     :', len(df[df['지역'].isna()]), '건')

50만원 이상 주문: 116 건
가전 + 온라인   : 104 건
수도권 또는 영남: 491 건
isin 사용       : 491 건
수량 3~5개      : 359 건
가전이 아닌 것  : 656 건
지역이 결측     : 30 건


In [13]:
# query() — 조건이 복잡할 때 읽기 쉬움
q = df.query('금액 >= 300000 and 카테고리 == "가전"')
print('query 결과:', len(q), '건')

# 파이썬 변수를 쓰려면 @
limit = 500_000
print('변수 사용 :', len(df.query('금액 >= @limit')), '건')

# 조건 + 열 선택을 한 번에 (loc 사용)
result = df.loc[df['금액'] >= 500_000, ['주문일', '지역', '카테고리', '금액']]
print('\n조건 + 열 선택:\n', result.head(3))

query 결과: 129 건
변수 사용 : 116 건

조건 + 열 선택:
           주문일  지역 카테고리           금액
4  2024-06-07  강원   가전 1,336,344.00
5  2024-11-09  충청   가전 1,760,591.00
20 2024-07-01  호남   가전 1,425,785.00


## 4.3 상위/하위 뽑기

In [14]:
print('금액 상위 5건:\n', df.nlargest(5, '금액')[['지역', '카테고리', '금액']])
print('\n금액 하위 3건:\n', df.nsmallest(3, '금액')[['지역', '카테고리', '금액']])
print('\n무작위 3건:\n', df.sample(3, random_state=0)[['지역', '카테고리', '금액']])

금액 상위 5건:
       지역 카테고리           금액
625  수도권   가전 2,656,689.00
389   영남   가전 2,050,422.00
441   호남   가전 2,008,097.00
207  수도권   가전 1,989,197.00
711   영남   가전 1,938,780.00

금액 하위 3건:
       지역 카테고리        금액
176   충청   식품 10,638.00
265   호남   식품 10,832.00
12   수도권   식품 11,534.00

무작위 3건:
       지역 카테고리         금액
613   강원   의류 153,616.00
202   충청   도서  76,916.00
55   수도권   도서  40,238.00


---

# Part 5. 데이터 정리

실제 데이터의 80%는 정리 작업입니다.

## 5.1 결측치

| 함수 | 하는 일 |
|---|---|
| `df.isna()` / `.notna()` | 결측 여부 (True/False) |
| `df.dropna()` | 결측 행 제거 |
| `df.fillna(값)` | 특정 값으로 채움 |
| `df['x'].fillna(df['x'].median())` | 통계값으로 채움 |
| `df.ffill()` / `.bfill()` | 앞/뒤 값으로 채움 (시계열에 유용) |
| `df.interpolate()` | 보간 |

In [15]:
work = df.copy()
print('처리 전 결측:\n', work.isna().sum()[lambda s: s > 0])

# 1) 범주형 결측 → '미상' 으로
work['지역'] = work['지역'].fillna('미상')

# 2) 수치형 결측 → 카테고리별 중앙값으로 (더 합리적)
work['단가'] = work['단가'].fillna(
    work.groupby('카테고리')['단가'].transform('median'))

# 3) 단가를 채웠으니 금액도 다시 계산
work['금액'] = work['단가'] * work['수량']

print('\n처리 후 결측:', work.isna().sum().sum(), '개')

# 참고: dropna 옵션
print('\ndropna() 전체 제거 :', df.dropna().shape)
print('특정 열만 기준     :', df.dropna(subset=['지역']).shape)
print('절반 이상 결측인 행만 제거:', df.dropna(thresh=df.shape[1] // 2).shape)

처리 전 결측:
 지역    30
단가    15
dtype: int64

처리 후 결측: 0 개

dropna() 전체 제거 : (765, 8)
특정 열만 기준     : (780, 8)
절반 이상 결측인 행만 제거: (810, 8)


> **결측치를 무조건 0으로 채우지 마세요.** '값이 없음'과 '값이 0'은 다릅니다. 평균이 왜곡됩니다.
> - 범주형 → `'미상'` 같은 별도 범주
> - 수치형 → 중앙값 (이상치에 강함), 또는 그룹별 중앙값
> - 시계열 → `ffill()`

## 5.2 중복 제거

In [16]:
print('중복 행 :', work.duplicated().sum(), '개')
print('중복 예시:\n', work[work.duplicated(keep=False)].head(4))

work = work.drop_duplicates()
print('\n제거 후 :', work.shape)

# 특정 열 기준 중복 (예: 고객당 첫 주문만)
first_order = work.sort_values('주문일').drop_duplicates(subset=['고객ID'], keep='first')
print('고객별 첫 주문 :', first_order.shape)

중복 행 : 10 개
중복 예시:
            주문일   지역 카테고리    채널  고객ID  수량         단가           금액
8   2024-03-14  수도권   의류   온라인  1121   4  38,787.00   155,148.00
84  2024-08-17  수도권   도서  오프라인  1108   5  20,614.00   103,070.00
175 2024-06-17   영남   가전   온라인  1188   3 246,790.00   740,370.00
242 2024-02-04  수도권   가전   온라인  1110   7 264,889.00 1,854,223.00

제거 후 : (800, 8)
고객별 첫 주문 : (196, 8)


## 5.3 타입 변환

| 함수 | 용도 |
|---|---|
| `astype('int64')` | 강제 변환 (실패하면 오류) |
| `pd.to_numeric(s, errors='coerce')` | 숫자로, 실패는 `NaN` |
| `pd.to_datetime(s)` | 날짜로 |
| `astype('category')` | **범주형 — 메모리 대폭 절약** |

In [17]:
messy = pd.Series(['1,200', '3,400', '없음', '5,600'])
clean = pd.to_numeric(messy.str.replace(',', ''), errors='coerce')
print('원본 :', messy.tolist())
print('변환 :', clean.tolist(), ' ← 실패한 값은 NaN')

# category 타입으로 메모리 절약
before = work.memory_usage(deep=True).sum()
for c in ['지역', '카테고리', '채널']:
    work[c] = work[c].astype('category')
after = work.memory_usage(deep=True).sum()

print(f'\n메모리 {before/1024:.0f} KB → {after/1024:.0f} KB '
      f'({(1-after/before)*100:.0f}% 절약)')
print('카테고리 값:', list(work['카테고리'].cat.categories))

원본 : ['1,200', '3,400', '없음', '5,600']
변환 : [1200.0, 3400.0, nan, 5600.0]  ← 실패한 값은 NaN

메모리 225 KB → 41 KB (82% 절약)
카테고리 값: ['가전', '도서', '식품', '의류']


## 5.4 문자열 처리 — `.str` 접근자

문자열 열에 `.str` 을 붙이면 파이썬 문자열 메서드를 **열 전체에** 적용할 수 있습니다.

| 메서드 | 하는 일 |
|---|---|
| `.str.strip()` | 앞뒤 공백 제거 |
| `.str.lower()` / `.upper()` | 대소문자 |
| `.str.replace(a, b)` | 치환 |
| `.str.contains('서울')` | 포함 여부 (조건에 사용) |
| `.str.startswith` / `endswith` | 시작·끝 |
| `.str.split('-')` | 분리 (`expand=True` 면 여러 열로) |
| `.str.len()` | 길이 |
| `.str[0:3]` | 슬라이싱 |
| `.str.extract(r'(\d+)')` | 정규식 추출 |
| `.str.zfill(5)` | 앞을 0으로 채움 |

In [18]:
raw = pd.DataFrame({
    '코드': [' A-1001 ', 'B-2002', ' c-3003', 'A-1004 '],
    '이름': ['김 철수', '이영희', ' 박민수', '최지은 '],
    '연락처': ['010-1234-5678', '010-2345-6789', '011-3456-7890', '010-4567-8901'],
})

out = pd.DataFrame()
out['코드'] = raw['코드'].str.strip().str.upper()
out['등급'] = out['코드'].str[0]
out['번호'] = out['코드'].str.split('-').str[1]
out['이름'] = raw['이름'].str.strip().str.replace(' ', '', regex=False)
out['성'] = out['이름'].str[0]
out['통신사'] = raw['연락처'].str[:3]
out['뒷자리'] = raw['연락처'].str.extract(r'-(\d{4})$')

print(out)
print('\nA등급만:\n', out[out['코드'].str.startswith('A')])

       코드 등급    번호   이름  성  통신사   뒷자리
0  A-1001  A  1001  김철수  김  010  5678
1  B-2002  B  2002  이영희  이  010  6789
2  C-3003  C  3003  박민수  박  011  7890
3  A-1004  A  1004  최지은  최  010  8901

A등급만:
        코드 등급    번호   이름  성  통신사   뒷자리
0  A-1001  A  1001  김철수  김  010  5678
3  A-1004  A  1004  최지은  최  010  8901


---

# Part 6. 열 만들기와 변형

## 6.1 새 열 만들기

In [19]:
w = work.copy()

# 1) 직접 대입 — 가장 단순
w['단가천원'] = w['단가'] / 1000

# 2) 조건부 — np.where (2분기)
w['고액여부'] = np.where(w['금액'] >= 300_000, '고액', '일반')

# 3) 여러 조건 — np.select
w['금액등급'] = np.select(
    [w['금액'] >= 500_000, w['금액'] >= 200_000, w['금액'] >= 50_000],
    ['S', 'A', 'B'], default='C')

# 4) 구간 나누기 — pd.cut (경계 지정) / pd.qcut (동일 개수)
w['수량구간'] = pd.cut(w['수량'], bins=[0, 2, 4, 8],
                       labels=['소량', '중량', '대량'])
w['금액분위'] = pd.qcut(w['금액'], q=4, labels=['하위25%', '중하', '중상', '상위25%'])

# 5) assign — 체이닝에 유용 (원본을 안 건드림)
w = w.assign(단가구간=lambda d: pd.cut(d['단가'], 3, labels=['저', '중', '고']))

print(w[['금액', '고액여부', '금액등급', '수량구간', '금액분위']].head())
print('\n금액등급 분포:\n', w['금액등급'].value_counts())

            금액 고액여부 금액등급 수량구간   금액분위
0    64,020.00   일반    B   중량     중하
1    89,190.00   일반    B   대량     중하
2    45,084.00   일반    C   중량  하위25%
3   115,346.00   일반    B   대량     중상
4 1,336,344.00   고액    S   대량  상위25%

금액등급 분포:
 금액등급
B    375
C    177
A    136
S    112
Name: count, dtype: int64


## 6.2 `map` · `apply` — 함수 적용

| 대상 | 함수 | 용도 |
|---|---|---|
| Series | `.map(딕셔너리 또는 함수)` | **원소별** 변환 |
| Series | `.apply(함수)` | 원소별 (map과 거의 같음) |
| DataFrame | `.apply(함수, axis=1)` | **행 단위** 처리 |
| DataFrame | `.apply(함수, axis=0)` | 열 단위 처리 |

> **`apply` 는 느립니다.** 내부적으로 파이썬 반복문입니다.
> 벡터 연산(`+`, `np.where`, `.str`)으로 표현할 수 있으면 그쪽이 수십 배 빠릅니다.

In [20]:
import time

# map — 딕셔너리로 값 바꾸기
region_en = {'수도권': 'Capital', '영남': 'Yeongnam', '호남': 'Honam',
             '충청': 'Chungcheong', '강원': 'Gangwon', '미상': 'Unknown'}
w['region_en'] = w['지역'].map(region_en)

# apply(axis=1) — 여러 열을 함께 봐야 할 때
w['비고'] = w.apply(
    lambda r: '대량고액' if (r['수량'] >= 5 and r['금액'] >= 300_000) else '-', axis=1)

print(w[['지역', 'region_en', '수량', '금액', '비고']].head())

# 속도 비교
big = w['금액'].repeat(20).reset_index(drop=True)
t0 = time.perf_counter(); r1 = big.apply(lambda v: v * 1.1);  t1 = time.perf_counter()
r2 = big * 1.1;                                                t2 = time.perf_counter()
print(f'\napply  : {t1-t0:.4f}초')
print(f'벡터화 : {t2-t1:.5f}초  ({(t1-t0)/(t2-t1):.0f}배 빠름)')

    지역    region_en  수량           금액    비고
0  수도권      Capital   4    64,020.00     -
1   충청  Chungcheong   6    89,190.00     -
2   호남        Honam   3    45,084.00     -
3  수도권      Capital   7   115,346.00     -
4   강원      Gangwon   6 1,336,344.00  대량고액

apply  : 0.0047초
벡터화 : 0.00029초  (16배 빠름)


## 6.3 열 이름·삭제·순서

In [21]:
tmp = w.rename(columns={'금액': 'amount', '수량': 'qty'})
print('이름 변경:', [c for c in tmp.columns if c in ('amount', 'qty')])

# 일괄 변경
print('공백 제거 예:', [c.replace(' ', '_') for c in ['a b', 'c d']])

# 삭제
print('\n열 삭제 후:', w.drop(columns=['비고', 'region_en']).shape)
print('행 삭제 후:', w.drop(index=[0, 1]).shape)

# 순서 바꾸기
cols = ['주문일', '지역', '카테고리', '금액']
print('\n원하는 순서만:\n', w[cols].head(2))

이름 변경: ['qty', 'amount']
공백 제거 예: ['a_b', 'c_d']

열 삭제 후: (800, 14)
행 삭제 후: (798, 16)

원하는 순서만:
          주문일   지역 카테고리        금액
0 2024-02-02  수도권   식품 64,020.00
1 2024-10-09   충청   식품 89,190.00


---

# Part 7. `groupby` — 이 교안의 핵심

## 7.1 개념: 쪼개고 → 계산하고 → 합친다

```
   [원본]              [Split]           [Apply]        [Combine]
  지역  금액          수도권: 100,300      수도권: 400
  수도권 100    →     영남  : 200      →  영남  : 200   →   지역   금액
  영남   200                                                수도권  400
  수도권 300                                                영남    200
```

> **`df.groupby('열')` 만 하면 아무 일도 안 일어납니다.** 뒤에 집계 함수를 붙여야 결과가 나옵니다.

In [22]:
g = w.groupby('지역', observed=True)
print('groupby 객체 :', type(g))
print('그룹 개수    :', g.ngroups)
print('그룹 크기    :\n', g.size())

print('\n지역별 금액 합계:\n', g['금액'].sum().round(0))
print('\n지역별 여러 통계:\n', g['금액'].agg(['count', 'sum', 'mean', 'median']).round(0))

groupby 객체 : <class 'pandas.core.groupby.generic.DataFrameGroupBy'>
그룹 개수    : 6
그룹 크기    :
 지역
강원      45
미상      30
수도권    307
영남     176
충청     117
호남     125
dtype: int64

지역별 금액 합계:
 지역
강원    14,155,703.00
미상     7,588,053.00
수도권   89,432,520.00
영남    51,471,204.00
충청    23,372,824.00
호남    29,637,681.00
Name: 금액, dtype: float64

지역별 여러 통계:
      count           sum       mean     median
지역                                            
강원      45 14,155,703.00 314,571.00 108,248.00
미상      30  7,588,053.00 252,935.00  57,258.00
수도권    307 89,432,520.00 291,311.00 108,188.00
영남     176 51,471,204.00 292,450.00 101,384.00
충청     117 23,372,824.00 199,768.00 100,190.00
호남     125 29,637,681.00 237,101.00 102,876.00


## 7.2 여러 열로 묶기

In [23]:
multi = w.groupby(['지역', '카테고리'], observed=True)['금액'].sum()
print('멀티인덱스 결과:\n', multi.head(8))
print('\n타입:', type(multi.index))

# 보기 좋게 펼치기
print('\nunstack 으로 표 형태:\n', multi.unstack().round(0))

# 인덱스를 열로 되돌리기
flat = w.groupby(['지역', '카테고리'], observed=True, as_index=False)['금액'].sum()
print('\nas_index=False:\n', flat.head(3))

멀티인덱스 결과:
 지역  카테고리
강원  가전     10,535,063.00
    도서        423,406.00
    식품        941,053.00
    의류      2,256,181.00
미상  가전      5,588,170.00
    도서        165,672.00
    식품        813,272.00
    의류      1,020,939.00
Name: 금액, dtype: float64

타입: <class 'pandas.core.indexes.multi.MultiIndex'>

unstack 으로 표 형태:
 카테고리            가전           도서           식품            의류
지역                                                        
강원   10,535,063.00   423,406.00   941,053.00  2,256,181.00
미상    5,588,170.00   165,672.00   813,272.00  1,020,939.00
수도권  63,760,195.00 3,198,085.00 6,414,359.00 16,059,880.00
영남   35,300,471.00 1,627,428.00 3,638,350.00 10,904,955.00
충청   11,994,838.00 2,068,625.00 2,292,562.00  7,016,798.00
호남   17,784,687.00 1,426,714.00 2,097,132.00  8,329,148.00

as_index=False:
    지역 카테고리            금액
0  강원   가전 10,535,063.00
1  강원   도서    423,406.00
2  강원   식품    941,053.00


## 7.3 `agg` — 열마다 다른 집계

**이름 있는 집계(named aggregation)** 가 가장 읽기 좋습니다.

```python
df.groupby('키').agg(
    새이름=('대상열', '집계함수'),
    ...
)
```

In [24]:
summary = w.groupby('지역', observed=True).agg(
    주문건수=('금액', 'count'),
    총매출=('금액', 'sum'),
    평균주문액=('금액', 'mean'),
    최대주문액=('금액', 'max'),
    총수량=('수량', 'sum'),
    고객수=('고객ID', 'nunique'),
    주력카테고리=('카테고리', lambda s: s.mode().iloc[0]),
).round(0)

summary['객단가'] = (summary['총매출'] / summary['고객수']).round(0)
summary = summary.sort_values('총매출', ascending=False)
print(summary)

     주문건수           총매출      평균주문액        최대주문액   총수량  고객수 주력카테고리        객단가
지역                                                                          
수도권   307 89,432,520.00 291,311.00 2,656,689.00  1212  153     식품 584,526.00
영남    176 51,471,204.00 292,450.00 2,050,422.00   730  120     의류 428,927.00
호남    125 29,637,681.00 237,101.00 2,008,097.00   461   89     의류 333,008.00
충청    117 23,372,824.00 199,768.00 1,842,218.00   473   93     의류 251,321.00
강원     45 14,155,703.00 314,571.00 1,688,876.00   180   43     식품 329,202.00
미상     30  7,588,053.00 252,935.00 1,570,362.00   110   28     식품 271,002.00


### 자주 쓰는 집계 함수

`'count'`(결측 제외), `'size'`(결측 포함), `'sum'`, `'mean'`, `'median'`, `'std'`, `'var'`, `'min'`, `'max'`, `'first'`, `'last'`, `'nunique'`, `'idxmax'`, `lambda`

## 7.4 `transform` — 그룹 통계를 원본 크기로 되돌리기

`agg` 는 그룹당 한 줄로 줄어듭니다. `transform` 은 **원본과 같은 길이**를 돌려줍니다. 그래서 새 열로 붙일 수 있습니다.

In [25]:
w['지역평균'] = w.groupby('지역', observed=True)['금액'].transform('mean')
w['지역대비'] = (w['금액'] / w['지역평균']).round(2)
w['지역내순위'] = w.groupby('지역', observed=True)['금액'].rank(ascending=False, method='min')
w['지역누적'] = w.sort_values('주문일').groupby('지역', observed=True)['금액'].cumsum()

print(w[['지역', '금액', '지역평균', '지역대비', '지역내순위']].head(6).round(0))

print('\nagg 결과 길이       :', len(w.groupby('지역', observed=True)['금액'].mean()))
print('transform 결과 길이 :', len(w.groupby('지역', observed=True)['금액'].transform('mean')))

    지역           금액       지역평균  지역대비  지역내순위
0  수도권    64,020.00 291,311.00  0.00 217.00
1   충청    89,190.00 199,768.00  0.00  68.00
2   호남    45,084.00 237,101.00  0.00  98.00
3  수도권   115,346.00 291,311.00  0.00 145.00
4   강원 1,336,344.00 314,571.00  4.00   2.00
5   충청 1,760,591.00 199,768.00  9.00   2.00

agg 결과 길이       : 6
transform 결과 길이 : 800


In [26]:
# filter — 조건을 만족하는 그룹 전체를 남기기
big_groups = w.groupby('지역', observed=True).filter(lambda g: g['금액'].sum() > 3e7)
print('매출 3천만 초과 지역만 :', big_groups['지역'].unique().tolist())
print('행 수:', len(big_groups), '/', len(w))

# 그룹별 상위 N개 뽑기 (자주 쓰는 패턴)
top2 = (w.sort_values('금액', ascending=False)
        .groupby('카테고리', observed=True)
        .head(2)[['카테고리', '지역', '금액']])
print('\n카테고리별 최고 주문 2건:\n', top2.sort_values('카테고리'))

매출 3천만 초과 지역만 : ['수도권', '영남']
행 수: 483 / 800

카테고리별 최고 주문 2건:
     카테고리   지역           금액
625   가전  수도권 2,656,689.00
389   가전   영남 2,050,422.00
85    도서   영남   171,696.00
134   도서  수도권   153,020.00
791   식품  수도권   137,942.00
141   식품   영남   135,170.00
9     의류   강원   389,095.00
221   의류   미상   387,975.00


## 7.5 `pivot_table` — 엑셀 피벗과 같은 것

In [27]:
pv = pd.pivot_table(w,
                    index='지역', columns='카테고리',
                    values='금액', aggfunc='sum',
                    fill_value=0, margins=True, margins_name='합계',
                    observed=True)
print((pv / 1e6).round(1), '\n(단위: 백만원)')

# 여러 집계를 동시에
pv2 = pd.pivot_table(w, index='채널', columns='카테고리',
                     values='금액', aggfunc=['count', 'mean'],
                     observed=True).round(0)
print('\n건수와 평균을 동시에:\n', pv2)

카테고리     가전   도서    식품    의류     합계
지역                                 
강원    10.50 0.40  0.90  2.30  14.20
미상     5.60 0.20  0.80  1.00   7.60
수도권   63.80 3.20  6.40 16.10  89.40
영남    35.30 1.60  3.60 10.90  51.50
충청    12.00 2.10  2.30  7.00  23.40
호남    17.80 1.40  2.10  8.30  29.60
합계   145.00 8.90 16.20 45.60 215.70 
(단위: 백만원)

건수와 평균을 동시에:
      count                     mean                               
카테고리    가전  도서   식품   의류         가전        도서        식품         의류
채널                                                                
오프라인    50  41   99   86 952,239.00 65,044.00 63,965.00 184,477.00
온라인    100  90  164  170 973,515.00 69,368.00 60,148.00 174,840.00


---

# Part 8. 표 합치기

| 함수 | 용도 |
|---|---|
| `pd.concat([a, b])` | **단순히 이어붙임** (위아래 또는 좌우) |
| `pd.merge(a, b, on='키')` | **키를 맞춰 조인** (SQL JOIN) |
| `a.join(b)` | 인덱스 기준 조인 |

## 8.1 `concat` — 이어붙이기

In [28]:
a = pd.DataFrame({'이름': ['철수', '영희'], '점수': [80, 90]})
b = pd.DataFrame({'이름': ['민수', '지은'], '점수': [70, 95]})

print('세로로 (axis=0):\n', pd.concat([a, b], ignore_index=True))

c = pd.DataFrame({'반': ['A', 'B']})
print('\n가로로 (axis=1):\n', pd.concat([a, c], axis=1))

# 출처 표시
print('\n출처 표시:\n', pd.concat([a, b], keys=['1반', '2반']))

세로로 (axis=0):
    이름  점수
0  철수  80
1  영희  90
2  민수  70
3  지은  95

가로로 (axis=1):
    이름  점수  반
0  철수  80  A
1  영희  90  B

출처 표시:
       이름  점수
1반 0  철수  80
   1  영희  90
2반 0  민수  70
   1  지은  95


## 8.2 `merge` — 키로 조인

```
   how='inner'   양쪽에 다 있는 것만        (교집합)
   how='left'    왼쪽 전부 + 맞는 오른쪽    ← 가장 많이 씀
   how='right'   오른쪽 전부
   how='outer'   양쪽 전부                  (합집합)
```

In [29]:
orders = pd.DataFrame({
    '주문번호': [1, 2, 3, 4, 5],
    '고객ID': [101, 102, 101, 103, 999],      # 999는 고객 목록에 없음
    '금액': [15000, 32000, 8000, 45000, 12000],
})
customers = pd.DataFrame({
    '고객ID': [101, 102, 103, 104],           # 104는 주문이 없음
    '이름': ['철수', '영희', '민수', '지은'],
    '등급': ['골드', '실버', '골드', 'VIP'],
})

for how in ['inner', 'left', 'right', 'outer']:
    m = pd.merge(orders, customers, on='고객ID', how=how)
    print(f"how='{how}'  →  {len(m)}행")
print()
print(pd.merge(orders, customers, on='고객ID', how='left'))

how='inner'  →  4행
how='left'  →  5행
how='right'  →  5행
how='outer'  →  6행

   주문번호  고객ID     금액   이름   등급
0     1   101  15000   철수   골드
1     2   102  32000   영희   실버
2     3   101   8000   철수   골드
3     4   103  45000   민수   골드
4     5   999  12000  NaN  NaN


In [30]:
# indicator=True — 어디서 온 행인지 확인 (조인 검증에 필수)
chk = pd.merge(orders, customers, on='고객ID', how='outer', indicator=True)
print(chk[['주문번호', '고객ID', '이름', '_merge']])
print('\n출처별 개수:\n', chk['_merge'].value_counts())

# 키 이름이 다를 때
left = pd.DataFrame({'id': [1, 2], 'v': ['a', 'b']})
right = pd.DataFrame({'key': [1, 2], 'w': ['x', 'y']})
print('\n키 이름이 다를 때:\n', pd.merge(left, right, left_on='id', right_on='key'))

# 같은 이름의 열이 양쪽에 있을 때
l2 = pd.DataFrame({'id': [1], '값': [10]})
r2 = pd.DataFrame({'id': [1], '값': [99]})
print('\nsuffixes:\n', pd.merge(l2, r2, on='id', suffixes=('_주문', '_고객')))

   주문번호  고객ID   이름      _merge
0  1.00   101   철수        both
1  3.00   101   철수        both
2  2.00   102   영희        both
3  4.00   103   민수        both
4   NaN   104   지은  right_only
5  5.00   999  NaN   left_only

출처별 개수:
 _merge
both          4
left_only     1
right_only    1
Name: count, dtype: int64

키 이름이 다를 때:
    id  v  key  w
0   1  a    1  x
1   2  b    2  y

suffixes:
    id  값_주문  값_고객
0   1    10    99


> **merge 후 반드시 행 수를 확인하세요.** 키가 중복이면 행이 **곱해져서 늘어납니다.**
> 아래처럼 검증하는 습관을 들이세요.
>
> ```python
> before = len(orders)
> m = orders.merge(customers, on='고객ID', how='left')
> assert len(m) == before, f'행이 {before} → {len(m)} 로 늘어남!'
> ```

In [31]:
# 키 중복으로 행이 폭발하는 예
dup = pd.DataFrame({'고객ID': [101, 101, 101], '메모': ['m1', 'm2', 'm3']})
print('주문 5행 + 메모 3행(같은 키) 조인 →',
      len(pd.merge(orders, dup, on='고객ID', how='left')), '행')
print('→ 고객 101의 주문 2건 × 메모 3건 = 6, 나머지 3건 = 총 9행')

# validate 로 미리 막기
try:
    pd.merge(orders, dup, on='고객ID', how='left', validate='m:1')
except Exception as e:
    print('\nvalidate 로 사전 차단 :', type(e).__name__)

주문 5행 + 메모 3행(같은 키) 조인 → 9 행
→ 고객 101의 주문 2건 × 메모 3건 = 6, 나머지 3건 = 총 9행

validate 로 사전 차단 : MergeError


---

# Part 9. 표 모양 바꾸기

```
[wide-form]                    [long-form (tidy)]
지역  가전  의류               지역   카테고리  금액
서울  100   200      ←→        서울    가전     100
부산   80   150                서울    의류     200
                               부산    가전      80
                               부산    의류     150
```

| 함수 | 방향 |
|---|---|
| `df.melt()` | **wide → long** (열을 행으로 녹임) |
| `df.pivot()` | **long → wide** (값을 열로 펼침) |
| `df.pivot_table()` | long → wide + **집계** (중복 키 허용) |
| `df.stack()` / `unstack()` | 인덱스 ↔ 열 |

> 시각화 라이브러리(seaborn, plotly)는 대부분 **long-form**을 원합니다. 사람이 보기엔 wide-form이 편합니다.

In [32]:
wide = pd.DataFrame({
    '지역': ['서울', '부산', '대구'],
    '2022': [100, 80, 60],
    '2023': [110, 85, 58],
    '2024': [125, 92, 63],
})
print('wide-form\n', wide)

long = wide.melt(id_vars='지역', var_name='연도', value_name='매출')
print('\nmelt → long-form\n', long.head(6))

back = long.pivot(index='지역', columns='연도', values='매출')
print('\npivot → 다시 wide-form\n', back)

wide-form
    지역  2022  2023  2024
0  서울   100   110   125
1  부산    80    85    92
2  대구    60    58    63

melt → long-form
    지역    연도   매출
0  서울  2022  100
1  부산  2022   80
2  대구  2022   60
3  서울  2023  110
4  부산  2023   85
5  대구  2023   58

pivot → 다시 wide-form
 연도  2022  2023  2024
지역                  
대구    60    58    63
부산    80    85    92
서울   100   110   125


In [33]:
# stack / unstack — 멀티인덱스 다루기
ms = w.groupby(['지역', '카테고리'], observed=True)['금액'].sum()
print('멀티인덱스 Series (long):\n', ms.head(4))

print('\nunstack() → 표로:\n', (ms.unstack() / 1e6).round(1))
print('\n다시 stack():\n', ms.unstack().stack().head(4))

# 어느 레벨을 펼칠지 지정
print('\nunstack(0) → 지역을 열로:\n', (ms.unstack(0) / 1e6).round(1))

멀티인덱스 Series (long):
 지역  카테고리
강원  가전     10,535,063.00
    도서        423,406.00
    식품        941,053.00
    의류      2,256,181.00
Name: 금액, dtype: float64

unstack() → 표로:
 카테고리    가전   도서   식품    의류
지역                        
강원   10.50 0.40 0.90  2.30
미상    5.60 0.20 0.80  1.00
수도권  63.80 3.20 6.40 16.10
영남   35.30 1.60 3.60 10.90
충청   12.00 2.10 2.30  7.00
호남   17.80 1.40 2.10  8.30

다시 stack():
 지역  카테고리
강원  가전     10,535,063.00
    도서        423,406.00
    식품        941,053.00
    의류      2,256,181.00
dtype: float64

unstack(0) → 지역을 열로:
 지역      강원   미상   수도권    영남    충청    호남
카테고리                                   
가전   10.50 5.60 63.80 35.30 12.00 17.80
도서    0.40 0.20  3.20  1.60  2.10  1.40
식품    0.90 0.80  6.40  3.60  2.30  2.10
의류    2.30 1.00 16.10 10.90  7.00  8.30


> **`pivot` 과 `pivot_table` 의 차이**
> - `pivot` : 인덱스+열 조합이 **유일해야** 함. 중복이면 오류.
> - `pivot_table` : 중복이 있으면 **집계**해 줌 (기본 평균). 실무에서는 이쪽을 더 많이 씁니다.

---

# Part 10. 시계열

## 10.1 날짜 타입으로 바꾸기

> **날짜를 문자열로 두면 아무것도 못 합니다.** `pd.to_datetime` 이 첫 단계입니다.

In [37]:
raw_dates = pd.Series(['2024-01-15', '2024/02/20', '2024년 3월 5일', '이상한값'])

print('기본 변환      :', pd.to_datetime(raw_dates[:2],format='mixed').tolist())
print('형식 지정      :', pd.to_datetime(['2024년 3월 5일'], format='%Y년 %m월 %d일').tolist())
print('오류는 NaT로   :', pd.to_datetime(raw_dates, errors='coerce', format='mixed').tolist())

print('\n원본 dtype :', w['주문일'].dtype)

기본 변환      : [Timestamp('2024-01-15 00:00:00'), Timestamp('2024-02-20 00:00:00')]
형식 지정      : [Timestamp('2024-03-05 00:00:00')]
오류는 NaT로   : [Timestamp('2024-01-15 00:00:00'), Timestamp('2024-02-20 00:00:00'), NaT, NaT]

원본 dtype : datetime64[ns]


## 10.2 `.dt` 접근자 — 날짜에서 부품 꺼내기

| 속성 | 값 |
|---|---|
| `.dt.year` / `.month` / `.day` | 연·월·일 |
| `.dt.hour` / `.minute` | 시·분 |
| `.dt.dayofweek` | 요일 (0=월 ~ 6=일) |
| `.dt.day_name()` | 요일 이름 |
| `.dt.quarter` | 분기 |
| `.dt.is_month_end` | 월말 여부 |
| `.dt.to_period('M')` | 월 단위로 (`2024-03`) |
| `.dt.strftime('%Y-%m')` | 문자열 서식 |

In [38]:
w['연'] = w['주문일'].dt.year
w['월'] = w['주문일'].dt.month
w['분기'] = w['주문일'].dt.quarter
w['요일번호'] = w['주문일'].dt.dayofweek
w['요일'] = w['주문일'].dt.day_name()
w['연월'] = w['주문일'].dt.to_period('M').astype(str)
w['주말'] = w['요일번호'] >= 5

print(w[['주문일', '월', '분기', '요일', '연월', '주말']].head())

print('\n요일별 매출(백만원):')
weekday_kr = ['월', '화', '수', '목', '금', '토', '일']
by_dow = w.groupby('요일번호', observed=True)['금액'].sum() / 1e6
for i, v in by_dow.items():
    print(f'  {weekday_kr[i]}요일 : {v:6.1f}')

print('\n주말 vs 평일:\n', (w.groupby('주말')['금액'].agg(['count', 'sum']) ))

         주문일   월  분기         요일       연월     주말
0 2024-02-02   2   1     Friday  2024-02  False
1 2024-10-09  10   4  Wednesday  2024-10  False
2 2024-08-26   8   3     Monday  2024-08  False
3 2024-06-09   6   2     Sunday  2024-06   True
4 2024-06-07   6   2     Friday  2024-06  False

요일별 매출(백만원):
  월요일 :   30.3
  화요일 :   27.8
  수요일 :   27.5
  목요일 :   37.2
  금요일 :   24.3
  토요일 :   32.2
  일요일 :   36.4

주말 vs 평일:
        count            sum
주말                         
False    564 147,104,998.00
True     236  68,552,986.00


## 10.3 `resample` — 시간 단위로 다시 묶기

**날짜를 인덱스로 설정한 뒤** 사용합니다. `groupby` 의 시간 버전입니다.

| 코드 | 단위 |
|---|---|
| `'D'` | 일 |
| `'W'` | 주 |
| `'ME'` | 월말 (구버전 `'M'`) |
| `'QE'` | 분기말 |
| `'YE'` | 연말 |
| `'h'`, `'min'` | 시, 분 |

In [39]:
ts = w.set_index('주문일').sort_index()

monthly = ts['금액'].resample('ME').sum() / 1e6
print('월별 매출(백만원):\n', monthly.round(1))

weekly = ts['금액'].resample('W').agg(['count', 'sum'])
print('\n주별 (앞 3주):\n', weekly.head(3))

# 날짜 슬라이싱 — 인덱스가 날짜면 이렇게 됩니다
print('\n3월 데이터 :', len(ts.loc['2024-03']), '건')
print('상반기     :', len(ts.loc['2024-01':'2024-06']), '건')

월별 매출(백만원):
 주문일
2024-01-31   11.30
2024-02-29   25.90
2024-03-31   22.50
2024-04-30   16.90
2024-05-31   15.00
2024-06-30   20.00
2024-07-31   16.80
2024-08-31   17.60
2024-09-30   18.00
2024-10-31   18.10
2024-11-30   16.00
2024-12-31   17.50
Freq: ME, Name: 금액, dtype: float64

주별 (앞 3주):
             count          sum
주문일                           
2024-01-07     11 1,614,840.00
2024-01-14     15 3,922,256.00
2024-01-21     10 2,735,226.00

3월 데이터 : 60 건
상반기     : 392 건


In [40]:
# 이동평균 · 증감 · 누적
daily = ts['금액'].resample('D').sum() / 1e6

calc = pd.DataFrame({'일매출': daily})
calc['7일이동평균'] = daily.rolling(7).mean()
calc['30일이동평균'] = daily.rolling(30, min_periods=1).mean()
calc['전일대비'] = daily.diff()
calc['전일대비율'] = daily.pct_change() * 100
calc['누적'] = daily.cumsum()
calc['전주동일'] = daily.shift(7)

print(calc.iloc[30:36].round(2))

print('\n월별 전월 대비 증감률(%):')
print((monthly.pct_change() * 100).round(1))

            일매출  7일이동평균  30일이동평균  전일대비    전일대비율    누적  전주동일
주문일                                                        
2024-02-01 1.20    0.35     0.41  0.60    99.10 12.50  0.24
2024-02-02 0.50    0.42     0.41 -0.70   -58.21 13.01  0.00
2024-02-03 0.55    0.46     0.42  0.05     9.66 13.56  0.26
2024-02-04 6.20    1.32     0.62  5.65 1,024.03 19.76  0.19
2024-02-05 0.28    1.34     0.62 -5.92   -95.41 20.04  0.13
2024-02-06 0.10    1.35     0.62 -0.19   -66.32 20.14  0.05

월별 전월 대비 증감률(%):
주문일
2024-01-31      NaN
2024-02-29   129.30
2024-03-31   -13.30
2024-04-30   -24.80
2024-05-31   -11.00
2024-06-30    32.90
2024-07-31   -16.20
2024-08-31     5.10
2024-09-30     2.50
2024-10-31     0.50
2024-11-30   -11.70
2024-12-31     9.30
Freq: ME, Name: 금액, dtype: float64


---

# Part 11. 흔한 실수와 성능

## 11.1 `SettingWithCopyWarning` — pandas 최대의 함정

```python
sub = df[df['금액'] > 100]     # 이게 사본인지 뷰인지 pandas도 모름
sub['새열'] = 1                # ← 경고! 원본이 바뀔 수도, 안 바뀔 수도
```

> **해결책 두 가지**
> 1. 부분집합을 만들 때 **`.copy()`** 를 붙인다 → `sub = df[cond].copy()`
> 2. 원본을 고칠 거면 **`.loc` 로 한 번에** → `df.loc[cond, '새열'] = 1`
>
> **절대 하지 말 것:** `df[cond]['열'] = 값` (연쇄 인덱싱). 조용히 실패합니다.

In [41]:
demo = pd.DataFrame({'a': [1, 2, 3, 4], 'b': [10, 20, 30, 40]})

# 나쁜 예 — 연쇄 인덱싱 (원본이 안 바뀜)
bad = demo.copy()
bad[bad['a'] > 2]['b'] = 999
print('연쇄 인덱싱 결과 — b가 안 바뀜:\n', bad)

# 좋은 예 1 — .loc 로 한 번에
good1 = demo.copy()
good1.loc[good1['a'] > 2, 'b'] = 999
print('\n.loc 사용 — 제대로 바뀜:\n', good1)

# 좋은 예 2 — 부분집합은 .copy()
sub = demo[demo['a'] > 2].copy()
sub['b'] = 999
print('\n.copy() 후 수정 — 원본은 그대로:\n', demo)

연쇄 인덱싱 결과 — b가 안 바뀜:
    a   b
0  1  10
1  2  20
2  3  30
3  4  40

.loc 사용 — 제대로 바뀜:
    a    b
0  1   10
1  2   20
2  3  999
3  4  999

.copy() 후 수정 — 원본은 그대로:
    a   b
0  1  10
1  2  20
2  3  30
3  4  40


/tmp/ipykernel_482/1603525702.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bad[bad['a'] > 2]['b'] = 999


## 11.2 흔한 실수 TOP 12

| # | 증상 | 원인 | 해결 |
|---|---|---|---|
| 1 | `SettingWithCopyWarning` | 연쇄 인덱싱 | `.loc` 또는 `.copy()` |
| 2 | `ValueError: truth value...` | `and` / `or` 사용 | `&`, `\|` + **괄호** |
| 3 | merge 후 행이 늘어남 | 키 중복 | `validate=`, 사전 `drop_duplicates` |
| 4 | 한글이 깨짐 | 인코딩 | `encoding='cp949'` 또는 `'utf-8-sig'` |
| 5 | 숫자가 문자열로 읽힘 | 천단위 콤마 | `thousands=','` 또는 `to_numeric` |
| 6 | 앞자리 0이 사라짐 | 자동 int 변환 | `dtype={'코드': str}` |
| 7 | 날짜 계산이 안 됨 | object 타입 | `pd.to_datetime` |
| 8 | 평균이 이상함 | 결측을 0으로 채움 | 중앙값 또는 그대로 두기 |
| 9 | `groupby` 후 인덱스가 이상 | 멀티인덱스 | `as_index=False` 또는 `reset_index()` |
| 10 | `apply` 가 느림 | 파이썬 반복문 | 벡터 연산 |
| 11 | 메모리 부족 | object 타입 | `category`, `usecols`, `dtype` 지정 |
| 12 | `inplace=True` 가 안 먹음 | 반환값이 None | **`inplace` 를 쓰지 말고 재대입** |

In [42]:
# 성능: 반복문 vs apply vs 벡터화
big = pd.DataFrame({'a': rng.random(200_000), 'b': rng.random(200_000)})

t0 = time.perf_counter()
r1 = [x + y for x, y in zip(big['a'], big['b'])]
t1 = time.perf_counter()
r2 = big.apply(lambda r: r['a'] + r['b'], axis=1)
t2 = time.perf_counter()
r3 = big['a'] + big['b']
t3 = time.perf_counter()

print(f'파이썬 반복문 : {t1-t0:.4f}초')
print(f'apply(axis=1) : {t2-t1:.4f}초')
print(f'벡터 연산     : {t3-t2:.5f}초')
print(f'\napply 대비 벡터화 : {(t2-t1)/(t3-t2):.0f}배 빠름')

파이썬 반복문 : 0.0449초
apply(axis=1) : 1.2834초
벡터 연산     : 0.00087초

apply 대비 벡터화 : 1469배 빠름


### 메서드 체이닝 — 읽기 좋은 코드

중간 변수를 만들지 않고 이어서 쓰는 방식입니다. 괄호로 감싸면 줄바꿈이 됩니다.

In [43]:
result = (
    df
    .dropna(subset=['지역'])
    .drop_duplicates()
    .assign(개당금액=lambda d: d['금액'] / d['수량'])
    .query('금액 > 50000')
    .groupby(['지역', '카테고리'], observed=True, as_index=False)
    .agg(건수=('금액', 'count'), 매출=('금액', 'sum'))
    .sort_values('매출', ascending=False)
    .head(8)
)
print(result.round(0))

     지역 카테고리  건수            매출
4   수도권   가전  65 63,743,467.00
8    영남   가전  35 35,300,471.00
16   호남   가전  20 17,784,687.00
7   수도권   의류  83 15,811,671.00
12   충청   가전  15 11,994,838.00
11   영남   의류  55 10,584,766.00
0    강원   가전  10 10,535,063.00
19   호남   의류  43  8,198,700.00


---

# Part 12. 종합 프로젝트 — 판매 데이터 분석

실제 분석 순서를 그대로 따라갑니다.

**1) 불러오기 → 2) 정리 → 3) 파생 열 → 4) 집계 → 5) 리포트**

In [44]:
# 1~3단계: 불러오기 + 정리 + 파생 열 (체이닝으로 한 번에)
raw = pd.read_csv('sales.csv', parse_dates=['주문일'], encoding='utf-8-sig')

clean = (
    raw
    .drop_duplicates()
    .assign(
        지역=lambda d: d['지역'].fillna('미상'),
        단가=lambda d: d['단가'].fillna(d.groupby('카테고리')['단가'].transform('median')),
    )
    .assign(
        금액=lambda d: d['단가'] * d['수량'],
        연월=lambda d: d['주문일'].dt.to_period('M').astype(str),
        분기=lambda d: 'Q' + d['주문일'].dt.quarter.astype(str),
        요일=lambda d: d['주문일'].dt.dayofweek,
        주말=lambda d: d['주문일'].dt.dayofweek >= 5,
    )
)

print(f'원본 {raw.shape} → 정리 후 {clean.shape}')
print('남은 결측:', clean.isna().sum().sum())
clean.head(3)

원본 (810, 8) → 정리 후 (800, 12)
남은 결측: 0


,주문일,지역,카테고리,채널,고객ID,수량,단가,금액,연월,분기,요일,주말
0,2024-02-02,수도권,식품,오프라인,1067,4,"16,005.00","64,020.00",2024-02,Q1,4,False
1,2024-10-09,충청,식품,온라인,1199,6,"14,865.00","89,190.00",2024-10,Q4,2,False
2,2024-08-26,호남,식품,오프라인,1001,3,"15,028.00","45,084.00",2024-08,Q3,0,False
